In [9]:
# # Cell 1 — Install dependencies (Kaggle)
# !pip -q install playwright pandas tldextract aiofiles
# !playwright install chromium
# ✅ One Kaggle cell to fix apt sources + install all Playwright deps
# (Paste into a PYTHON cell)

# 0) Disable the problematic r2u repo if it exists (safe no-op if not present)
!set -e; \
  if ls /etc/apt/sources.list.d/* 1>/dev/null 2>&1; then \
    for f in /etc/apt/sources.list.d/*.list; do \
      if [ -f "$f" ] && grep -qi "r2u.stat.illinois.edu" "$f"; then \
        echo "Disabling repo file: $f"; \
        mv "$f" "$f.disabled"; \
      fi; \
    done; \
  fi

# 1) Update apt
!apt-get update -y >/dev/null

# 2) Install Linux libraries Chromium needs
# NOTE: libatk1.0-0 is the correct package name for libatk-1.0.so.0
!apt-get install -y \
  libatk1.0-0 \
  libatk-bridge2.0-0 \
  libcups2 \
  libxkbcommon0 \
  libxcomposite1 \
  libxdamage1 \
  libxrandr2 \
  libgbm1 \
  libpango-1.0-0 \
  libpangocairo-1.0-0 \
  libcairo2 \
  libasound2 \
  libnss3 \
  libxss1 \
  libgtk-3-0 \
  libx11-xcb1 \
  libxext6 \
  libxfixes3 \
  libxi6 \
  libdrm2 \
  >/dev/null

# 3) Python deps + Playwright browser
!pip install -q playwright pandas tldextract aiofiles nest_asyncio
!playwright install chromium

!python -m pip install -U playwright
!python -m playwright install

print("✅ System + Python + Playwright Chromium installed.")

zsh:1: no matches found: /etc/apt/sources.list.d/*
zsh:1: command not found: apt-get
zsh:1: command not found: apt-get
zsh:1: command not found: playwright
Defaulting to user installation because normal site-packages is not writeable
(node:12879) [DEP0169] DeprecationWarning: `url.parse()` behavior is not standardized and prone to errors that have security implications. Use the WHATWG URL API instead. CVEs are not issued for `url.parse()` vulnerabilities.
(Use `node --trace-deprecation ...` to show where the warning was created)
Error: socket hang up
    at TLSSocket.socketOnEnd (node:_http_client:599:25)
    at TLSSocket.emit (node:events:520:35)
    at endReadableNT (node:internal/streams/readable:1701:12)
    at process.processTicksAndRejections (node:internal/process/task_queues:89:21) {
  code: 'ECONNRESET'
}
(node:12998) [DEP0169] DeprecationWarning: `url.parse()` behavior is not standardized and prone to errors that have security implications. Use the WHATWG URL API instead. CVE

In [10]:
import shutil, os

OUT_DIR = "./out_gtm"   # must match what you pass to crawl_sites_from_csv

if os.path.exists(OUT_DIR):
    shutil.rmtree(OUT_DIR)
os.makedirs(OUT_DIR, exist_ok=True)

print("✅ Cleared out_gtm (removed processed.txt + old manifest).")

✅ Cleared out_gtm (removed processed.txt + old manifest).


In [11]:
import nest_asyncio
nest_asyncio.apply()

In [12]:
from playwright.async_api import async_playwright

CONSENTOMATIC_DIR = "/Users/muhammadhamdazam/Documents/Spring 2026/Eman/crux/Consent-O-Matic/Extension"
USER_DATA_DIR = "./chrome_profile_consentomatic"

async def test_extension():
    async with async_playwright() as p:
        context = await p.chromium.launch_persistent_context(
            user_data_dir=USER_DATA_DIR,
            headless=False,  # extensions need headful
            args=[
                f"--disable-extensions-except={CONSENTOMATIC_DIR}",
                f"--load-extension={CONSENTOMATIC_DIR}",
                "--no-first-run",
                "--no-default-browser-check",
            ],
        )
        page = await context.new_page()
        await page.goto("https://example.com", wait_until="domcontentloaded")
        await page.wait_for_timeout(5000)
        await context.close()

# In notebook: await test_extension()

In [13]:
# Cell 2 — Imports + helpers
import asyncio
import os
import re
import json
import time
import hashlib
from typing import Optional, Dict, Any, List, Set, Tuple

import pandas as pd
import tldextract
import aiofiles

from playwright.async_api import async_playwright, TimeoutError as PWTimeoutError

GTM_URL_RE = re.compile(r"gtm", re.IGNORECASE)

from urllib.parse import urlparse

def is_gtm_library_url(url: str) -> bool:
    """
    True only for GTM/GTAG *library* scripts, not GA collect/event endpoints.
    """
    if not url:
        return False

    u = url.lower()

    # Only accept googletagmanager script endpoints
    # (This is the main thing you asked for)
    if "googletagmanager.com" in u:
        p = urlparse(u).path

        # GTM container loader
        if p.endswith("/gtm.js"):
            return True

        # gtag library loader
        if p.endswith("/gtag/js"):
            return True

        # sometimes appears as /gtag/js?id=...
        if "/gtag/js" in p:
            return True

    return False

def normalize_site(raw: str) -> str:
    raw = (raw or "").strip()
    raw = raw.replace(" ", "")
    return raw

def site_candidates(site: str) -> List[str]:
    if site.startswith("http://") or site.startswith("https://"):
        return [site]
    return [f"https://{site}", f"http://{site}"]

def safe_dirname(url_or_site: str) -> str:
    ext = tldextract.extract(url_or_site)
    host = ".".join([p for p in [ext.subdomain, ext.domain, ext.suffix] if p]).strip(".")
    if not host:
        host = "unknown"
    h = hashlib.sha1(url_or_site.encode("utf-8", errors="ignore")).hexdigest()[:10]
    return f"{host}_{h}"

async def save_bytes(path: str, data: bytes) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    async with aiofiles.open(path, "wb") as f:
        await f.write(data)

async def append_jsonl(path: str, obj: Dict[str, Any]) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    async with aiofiles.open(path, "a", encoding="utf-8") as f:
        await f.write(json.dumps(obj, ensure_ascii=False) + "\n")

async def maybe_click_cookie_accept(page) -> None:
    """
    Kaggle-safe best-effort cookie acceptor.
    Not perfect, but helps with lots of CMPs.
    """
    selectors = [
        "button:has-text('Accept all')",
        "button:has-text('Accept All')",
        "button:has-text('ACCEPT ALL')",
        "button:has-text('Accept')",
        "button:has-text('I agree')",
        "button:has-text('I Agree')",
        "button:has-text('Agree')",
        "button:has-text('Allow all')",
        "button:has-text('Allow All')",
        "button:has-text('OK')",
        "button:has-text('Got it')",
        "button:has-text('Continue')",
        "text=Accept all",
        "text=I agree",
        "text=Agree",
        # Common CMP IDs/classes
        "#onetrust-accept-btn-handler",
        "button#onetrust-accept-btn-handler",
        "button.ot-sdk-container",
        "button[aria-label*='accept' i]",
        "button[class*='accept' i]",
        "button[id*='accept' i]",
    ]
    for sel in selectors:
        try:
            loc = page.locator(sel).first
            if await loc.count() > 0 and await loc.is_visible():
                await loc.click(timeout=1500)
                return
        except Exception:
            continue

async def load_processed(processed_path: str) -> Set[str]:
    if not os.path.exists(processed_path):
        return set()
    out = set()
    async with aiofiles.open(processed_path, "r", encoding="utf-8") as f:
        async for line in f:
            s = line.strip()
            if s:
                out.add(s)
    return out

async def mark_processed(processed_path: str, site: str) -> None:
    os.makedirs(os.path.dirname(processed_path), exist_ok=True)
    async with aiofiles.open(processed_path, "a", encoding="utf-8") as f:
        await f.write(site + "\n")

In [14]:
# # Cell 3 — Core crawl function (captures GTM responses + saves bodies if possible)
# from urllib.parse import urlparse

# # def is_gtm_library_url(url: str) -> bool:
# #     if not url:
# #         return False
# #     u = url.lower()
# #     if "googletagmanager.com" not in u:
# #         return False
# #     p = urlparse(u).path
# #     return p.endswith("/gtm.js") or p.endswith("/gtag/js") or "/gtag/js" in p

async def fetch_body_fallback(context, url: str) -> bytes | None:
    try:
        r = await context.request.get(url, timeout=15000)
        if not r.ok:
            return None
        return await r.body()
    except Exception:
        return None

# async def crawl_one_site(
#     context,
#     site: str,
#     out_dir: str,
#     manifest_path: str,
#     nav_timeout_ms: int = 20000,
#     resource_timeout_ms: int = 10000,
# ) -> tuple[bool, str]:
#     site = normalize_site(site)
#     if not site:
#         return False, "empty_site"

#     folder = os.path.join(out_dir, safe_dirname(site))
#     os.makedirs(folder, exist_ok=True)

#     page = await context.new_page()
#     seen_urls = set()  # ✅ dedupe per site

#     async def record_gtm(url: str, status=None, ctype=None, saved_path=None, err=None):
#         if url in seen_urls:
#             return
#         seen_urls.add(url)

#         await append_jsonl(
#             manifest_path,
#             {
#                 "site": site,
#                 "page_url": page.url if page.url else "",
#                 "resource_url": url,
#                 "status": status,
#                 "content_type": ctype,
#                 "saved_path": saved_path,
#                 "error": err,
#                 "timestamp_unix": time.time(),
#             },
#         )

#     async def on_response(response):
#         try:
#             url = response.url or ""
#             if not is_gtm_library_url(url):
#                 return

#             headers = response.headers or {}
#             ctype = headers.get("content-type") or headers.get("Content-Type")

#             status = None
#             try:
#                 status = response.status
#             except Exception:
#                 pass

#             body = None
#             err = None

#             # 1) normal body()
#             try:
#                 body = await asyncio.wait_for(response.body(), timeout=resource_timeout_ms / 1000)
#             except Exception as e:
#                 err = f"body_read_failed:{type(e).__name__}"

#             # 2) fallback fetch
#             if body is None:
#                 fb = await fetch_body_fallback(context, url)
#                 if fb:
#                     body = fb
#                     err = None

#             saved_path = None
#             if body:
#                 name_hash = hashlib.sha1(url.encode("utf-8", errors="ignore")).hexdigest()[:16]
#                 saved_path = os.path.join(folder, f"gtm_{name_hash}.js")
#                 try:
#                     await save_bytes(saved_path, body)
#                 except Exception as e:
#                     saved_path = None
#                     err = f"save_failed:{type(e).__name__}"

#             await record_gtm(url, status=status, ctype=ctype, saved_path=saved_path, err=err)

#         except Exception:
#             return

#     page.on("response", on_response)

#     final_url = None
#     last_err = None

#     try:
#         for candidate in site_candidates(site):
#             try:
#                 await page.goto(candidate, wait_until="domcontentloaded", timeout=nav_timeout_ms)

#                 # ✅ Give Consent-O-Matic time to click + page time to load tags
#                 await page.wait_for_timeout(6000)

#                 # optional fallback clicker (keep it)
#                 await maybe_click_cookie_accept(page)
#                 await page.wait_for_timeout(3000)

#                 final_url = page.url
#                 last_err = None
#                 break
#             except PWTimeoutError:
#                 last_err = "nav_timeout"
#             except Exception as e:
#                 last_err = f"nav_failed:{type(e).__name__}"

#         if not final_url:
#             return False, last_err or "nav_failed"

#         # Trigger lazy loads
#         try:
#             await page.evaluate("""() => window.scrollTo(0, Math.min(document.body.scrollHeight, 3000))""")
#         except Exception:
#             pass

#         await page.wait_for_timeout(4000)

#         # ✅ DOM scan to catch script tags even if response body is not readable
#         try:
#             script_srcs = await page.eval_on_selector_all(
#                 "script[src]",
#                 "els => els.map(e => e.src).filter(Boolean)"
#             )
#             for s in script_srcs:
#                 if is_gtm_library_url(s):
#                     # Try to download via request if we haven't already
#                     if s not in seen_urls:
#                         body = await fetch_body_fallback(context, s)
#                         saved_path = None
#                         err = None
#                         if body:
#                             name_hash = hashlib.sha1(s.encode("utf-8", errors="ignore")).hexdigest()[:16]
#                             saved_path = os.path.join(folder, f"gtm_{name_hash}.js")
#                             await save_bytes(saved_path, body)
#                         await record_gtm(s, status=200, ctype="from_dom_script_tag", saved_path=saved_path, err=err)
#         except Exception:
#             pass

#         return True, final_url

#     finally:
#         try:
#             await page.close()
#         except Exception:
#             pass

async def crawl_one_site(
    context,
    site: str,
    out_dir: str,
    manifest_path: str,
    nav_timeout_ms: int = 20000,
    resource_timeout_ms: int = 10000,
) -> tuple[bool, str]:
    site = normalize_site(site)
    if not site:
        return False, "empty_site"

    folder = os.path.join(out_dir, safe_dirname(site))
    os.makedirs(folder, exist_ok=True)

    page = await context.new_page()
    seen_urls = set()

    # ✅ FIX 1: page_url passed explicitly at call time, not captured lazily
    async def record_gtm(url: str, page_url: str, status=None, ctype=None, saved_path=None, err=None):
        if url in seen_urls:
            return
        seen_urls.add(url)
        await append_jsonl(
            manifest_path,
            {
                "site": site,
                "page_url": page_url,
                "resource_url": url,
                "status": status,
                "content_type": ctype,
                "saved_path": saved_path,
                "error": err,
                "timestamp_unix": time.time(),
            },
        )

    async def on_response(response):
        try:
            url = response.url or ""
            if not is_gtm_library_url(url):
                return

            # ✅ FIX 2: snapshot page_url immediately when response fires
            page_url = page.url or ""

            headers = response.headers or {}
            ctype = headers.get("content-type") or headers.get("Content-Type")
            status = None
            try:
                status = response.status
            except Exception:
                pass

            body = None
            err = None

            try:
                body = await asyncio.wait_for(response.body(), timeout=resource_timeout_ms / 1000)
            except Exception as e:
                err = f"body_read_failed:{type(e).__name__}"

            if body is None:
                fb = await fetch_body_fallback(context, url)
                if fb:
                    body = fb
                    err = None

            saved_path = None
            if body:
                name_hash = hashlib.sha1(url.encode("utf-8", errors="ignore")).hexdigest()[:16]
                saved_path = os.path.join(folder, f"gtm_{name_hash}.js")
                try:
                    await save_bytes(saved_path, body)
                except Exception as e:
                    saved_path = None
                    err = f"save_failed:{type(e).__name__}"

            await record_gtm(url, page_url=page_url, status=status, ctype=ctype, saved_path=saved_path, err=err)

        except Exception:
            return

    page.on("response", on_response)

    final_url = None
    last_err = None

    try:
        for candidate in site_candidates(site):
            try:
                await page.goto(candidate, wait_until="domcontentloaded", timeout=nav_timeout_ms)

                # ✅ FIX 3: wait for networkidle so post-consent GTM fires are captured
                # This replaces the hardcoded 6000ms sleep
                try:
                    await page.wait_for_load_state("networkidle", timeout=8000)
                except Exception:
                    await page.wait_for_timeout(4000)

                # Fallback clicker for CMPs Consent-o-matic missed
                await maybe_click_cookie_accept(page)

                # Wait again after any consent click — GTM may fire post-consent
                try:
                    await page.wait_for_load_state("networkidle", timeout=5000)
                except Exception:
                    await page.wait_for_timeout(2000)

                final_url = page.url
                last_err = None
                break
            except PWTimeoutError:
                last_err = "nav_timeout"
            except Exception as e:
                last_err = f"nav_failed:{type(e).__name__}"

        if not final_url:
            return False, last_err or "nav_failed"

        # Trigger lazy loads
        try:
            await page.evaluate("() => window.scrollTo(0, Math.min(document.body.scrollHeight, 3000))")
        except Exception:
            pass

        # Wait for any lazy-loaded GTM tags to fire after scroll
        try:
            await page.wait_for_load_state("networkidle", timeout=5000)
        except Exception:
            await page.wait_for_timeout(3000)

        # DOM scan — catches script tags where response body was unreadable
        try:
            script_srcs = await page.eval_on_selector_all(
                "script[src]",
                "els => els.map(e => e.src).filter(Boolean)"
            )
            current_page_url = page.url  # snapshot once for all DOM hits
            for s in script_srcs:
                if is_gtm_library_url(s) and s not in seen_urls:
                    body = await fetch_body_fallback(context, s)
                    saved_path = None
                    err = None
                    if body:
                        name_hash = hashlib.sha1(s.encode("utf-8", errors="ignore")).hexdigest()[:16]
                        saved_path = os.path.join(folder, f"gtm_{name_hash}.js")
                        await save_bytes(saved_path, body)
                    await record_gtm(s, page_url=current_page_url, status=200, ctype="from_dom_script_tag", saved_path=saved_path, err=err)
                    # Debug: log what the page actually looks like
                    try:
                        title = await page.title()
                        html_snippet = await page.evaluate("() => document.body?.innerText?.slice(0, 200)")
                        print(f"[DEBUG] {site} | title={title!r} | body={html_snippet!r}")
                    except Exception:
                        pass
        except Exception:
            pass

        return True, final_url

    finally:
        try:
            await page.close()
        except Exception:
            pass

In [15]:
async def crawl_sites_from_csv(
    csv_path: str,
    column: str,
    out_dir: str = "./out_gtm",
    start_row: int = 0,
    end_row: int = None,
    concurrency: int = 4,
    nav_timeout_ms: int = 25000,
    resource_timeout_ms: int = 10000,
    consentomatic_dir: str = None,
    user_data_dir: str = "./chrome_profile_consentomatic",
):
    
    os.makedirs(out_dir, exist_ok=True)
    processed_path = os.path.join(out_dir, "processed.txt")
    manifest_path = os.path.join(out_dir, "gtm_manifest.jsonl")
    results_path = os.path.join(out_dir, "site_results.jsonl")

    df = pd.read_csv(csv_path)
    if column not in df.columns:
        raise ValueError(f"Column '{column}' not found. Available: {list(df.columns)}")

    df = df.iloc[start_row:end_row]
    sites = [normalize_site(str(x)) for x in df[column].tolist()]
    sites = [s for s in sites if s]

    print(f"Processing rows {start_row} → {end_row}")
    print(f"Total sites in this batch: {len(sites)}")

    processed = await load_processed(processed_path)
    todo = [s for s in sites if s not in processed]
    print(f"After resume filter: {len(todo)} sites left")

    # sem = asyncio.Semaphore(concurrency)
    # Drop concurrency to 1 when extension is loaded — shared persistent context
    # can't handle parallel tabs with extension state reliably
    effective_concurrency = 1 if consentomatic_dir else concurrency
    sem = asyncio.Semaphore(effective_concurrency)

    async with async_playwright() as p:
        # ✅ Use a persistent context so extensions work
        args = [
            "--no-first-run",
            "--no-default-browser-check",
            "--disable-dev-shm-usage",
        ]

        if consentomatic_dir:
            args += [
                f"--disable-extensions-except={consentomatic_dir}",
                f"--load-extension={consentomatic_dir}",
            ]

        # IMPORTANT: headless must be False for extensions
        context = await p.chromium.launch_persistent_context(
            user_data_dir=user_data_dir,
            headless=False,
            args=args,
            viewport={"width": 1280, "height": 800},
        )

        async def worker(site: str):
            async with sem:
                ok, info = await crawl_one_site(
                    context=context,
                    site=site,
                    out_dir=out_dir,
                    manifest_path=manifest_path,
                    nav_timeout_ms=nav_timeout_ms,
                    resource_timeout_ms=resource_timeout_ms,
                )

                await append_jsonl(
                    results_path,
                    {"site": site, "ok": ok, "info": info, "timestamp_unix": time.time()},
                )
                await mark_processed(processed_path, site)

        tasks = [asyncio.create_task(worker(s)) for s in todo]
        for coro in asyncio.as_completed(tasks):
            try:
                await coro
            except Exception as e:
                print(f"[ERROR] worker crashed: {type(e).__name__}: {e}")

        await context.close()

    print("Done.")
    print("Manifest:", manifest_path)

In [16]:
# Cell 5 — Run it (EDIT THESE TWO)
CSV_PATH = "/Users/muhammadhamdazam/Documents/Spring 2026/Eman/crux/202601.csv"  # <-- change
COLUMN_NAME = "origin"                                     # <-- change (e.g. "domain")
df = pd.read_csv(CSV_PATH)
print(df.iloc[0:10][COLUMN_NAME].tolist())
# Tip: start with small limit to test
await crawl_sites_from_csv(
    csv_path=CSV_PATH,
    column=COLUMN_NAME,
    out_dir="./out_gtm",
    start_row=0,
    end_row=10,
    concurrency=4,
    consentomatic_dir=CONSENTOMATIC_DIR,
    user_data_dir=USER_DATA_DIR,
)

['https://bbs.animanch.com', 'https://asndigital.bkn.go.id', 'https://darkero.com', 'https://auto.bazos.sk', 'https://archive.org', 'https://archiveofourown.org', 'https://accounts.google.com', 'https://accounts.nintendo.com', 'https://hochi.news', 'https://jp.xhamster.com']
Processing rows 0 → 10
Total sites in this batch: 10
After resume filter: 10 sites left
Done.
Manifest: ./out_gtm/gtm_manifest.jsonl


In [17]:
# Cell 6 — Quick inspection: how many GTM hits did we collect?
import json

manifest_path = "/Users/muhammadhamdazam/Documents/Spring 2026/Eman/crux/crawler/out_gtm/gtm_manifest.jsonl"
count = 0
unique_sites = set()
unique_urls = set()

if os.path.exists(manifest_path):
    with open(manifest_path, "r", encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)
            count += 1
            unique_sites.add(obj.get("site"))
            unique_urls.add(obj.get("resource_url"))

print("Total GTM-like network hits:", count)
print("Unique sites with hits:", len(unique_sites))
print("Unique resource URLs:", len(unique_urls))

# show a few sample urls
for i, u in enumerate(list(unique_urls)[:20]):
    print("-", u)

Total GTM-like network hits: 8
Unique sites with hits: 6
Unique resource URLs: 8
- https://www.googletagmanager.com/gtm.js?id=GTM-TLDPV3J
- https://www.googletagmanager.com/gtag/js?id=G-T4L4FCFNJ3&cx=c&gtm=4e62j0
- https://www.googletagmanager.com/gtag/js?id=UA-204930982-1
- https://www.googletagmanager.com/gtm.js?id=GTM-5CQT2R2
- https://www.googletagmanager.com/gtag/js?id=G-HLZSNE9Z0C
- https://www.googletagmanager.com/gtag/js?id=G-TR47ZLJGXH
- https://www.googletagmanager.com/gtag/js?id=G-VVXF4DLZVB&cx=c&gtm=4e62j0
- https://www.googletagmanager.com/gtag/js?id=G-D3VMVHCY1Y


In [18]:
# Cell 7 — Optional: export a clean CSV of GTM URLs (site -> gtm_url)
import pandas as pd
import json

# manifest_path = "/Users/emannabeel/Documents/gtm_clean/crux/crawler/out_gtm/gtm_manifest.jsonl"  # <-- change if needed
rows = []

with open(manifest_path, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        rows.append({
            "site": obj.get("site"),
            "page_url": obj.get("page_url"),
            "gtm_resource_url": obj.get("resource_url"),
            "status": obj.get("status"),
            "saved_path": obj.get("saved_path"),
            "error": obj.get("error"),
        })

out_csv = "/Users/muhammadhamdazam/Documents/Spring 2026/Eman/crux/crawler/gtm_urls_new.csv"
pd.DataFrame(rows).drop_duplicates().to_csv(out_csv, index=False)
out_csv

'/Users/muhammadhamdazam/Documents/Spring 2026/Eman/crux/crawler/gtm_urls_new.csv'